In [ ]:
# Рекомендую в отдельном виртуальном окружении
pip install "autogluon.tabular[all]"  # для табличных данных (самое популярное)
# или просто
pip install autogluon

In [ ]:
# baseline_autogluon.py
import pandas as pd
from autogluon.tabular import TabularDataset, TabularPredictor
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 1. Загружаем данные (пример с классификацией)
# Замени на свои пути или свои датафреймы
train_data = TabularDataset('train.csv')          # или pd.read_csv(...)
test_data  = TabularDataset('test.csv')

# Если у тебя один файл и нужно поделить самому:
# train_data, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['target'])

# 2. Указываем название целевой переменной
label = 'target'  # ←←←←← замени на своё название колонки

# 3. Запускаем AutoML (всё остальное делает AutoGluon)
predictor = TabularPredictor(
    label=label,
    problem_type=None,          # сам определит (classification / regression)
    eval_metric=None,           # сам выберет лучшую метрику
    path='ag_models'            # куда сохранит модели
).fit(
    train_data,
    presets='best_quality',     # ← самый сильный бейзлайн (почти всегда топ на Kaggle)
    # presets='good_quality'    # если мало времени/памяти
    time_limit=60*60*2,         # 2 часа максимум (можно 3600, 7200, None=без лимита)
    # hyperparameters='multimodal',  # если есть текст/изображения в данных
    ag_args_fit={'num_gpus': 1} if __import__('torch').cuda.is_available() else {}
)

# 4. Смотрим результаты на тренировочных данных
print(predictor.leaderboard(train_data, silent=True))

# 5. Предсказания на тесте
preds = predictor.predict(test_data)                    # просто классы / значения
preds_proba = predictor.predict_proba(test_data)        # вероятности (для классификации)

# 6. Если нужно сохранить сабмит
submission = test_data[['id']].copy()  # если есть колонка id
submission['target'] = preds
submission.to_csv('submission_autogluon.csv', index=False)
print('Готово! submission_autogluon.csv сохранён')

In [ ]:
presets='best_quality'      # почти всегда топ-1..топ-5 на лидерборде (2–8 часов)
presets='high_quality'      # чуть быстрее
presets='good_quality'      # хороший результат за 15–40 минут
presets='medium_quality'    # если совсем мало времени

In [ ]:
pip install "autogluon[multimodal]"      # это новая унифицированная версия
# или старый способ (до 1.0 был отдельно)
# pip install autogluon.tabular autogluon.vision autogluon.text

In [ ]:
import pandas as pd
from autogluon.multimodal import MultiModalPredictor
from autogluon.tabular import TabularDataset

# ------------------- ЗАГРУЗКА ДАННЫХ -------------------
# У тебя может быть один csv, где:
# - есть колонка с текстом (например "description", "review", "title")
# - есть пути к изображениям (например "image_path")
# - остальные табличные признаки (цена, категория и т.д.)
# - и целевая переменная

train_df = pd.read_csv('train.csv')   # или TabularDataset('train.csv')
test_df  = pd.read_csv('test.csv')

# Важно: пути к изображениям должны быть абсолютными или относительными, но существующими!
# Пример строки:
# id | title                | description               | price | category | image_path          | label
# 1  | Красивый свитер      | Тёплый шерстяной...      | 3500  | одежда   | images/001.jpg      | positive

# ------------------- ОБУЧЕНИЕ -------------------
predictor = MultiModalPredictor(
    label='label',                    # название твоей целевой колонки
    problem_type=None,                # сам поймёт (classification/regression)
    eval_metric=None,                 # сам выберёт лучшую
    path='ag_multimodal_models'
)

predictor.fit(
    train_data=train_df,
    presets='best_quality',           # или 'high_quality', 'medium_quality'
    time_limit=60*60*4,               # 4 часа — обычно хватает для топ-результата
    # hyperparameters='multimodal_default',  # дефолтный стек (BERT + ResNet и т.д.)
)

# Готово! AutoGluon внутри запустит:
# - предобученный multilingual BERT / DeBERTa-v3 / RoBERTa для текста
# - ResNet-50 / ViT / Swin-Transformer для изображений
# - LightGBM/XGBoost/ CatBoost + нейронки для табличных данных
# - и всё это зафьюзит в одну сильную модель

In [ ]:
# Предсказание на тесте
preds = predictor.predict(test_df)                    # классы / значения
proba = predictor.predict_proba(test_df)              # вероятности

# Сохранение сабмита
submission = test_df[['id']].copy()
submission['label'] = preds
submission.to_csv('submission_multimodal.csv', index=False)

In [ ]:
predictor.fit(
    train_data=train_df,
    presets='best_quality',
    
    # Какие колонки что
    text_columns=['title', 'description'],        # список колонок с текстом
    image_columns=['image_path', 'photo2'],       # несколько фото можно!
    
    # Какие модели использовать
    hyperparameters={
        "model.hf_text.checkpoint_name": "cointegrated/rubert-tiny2",  # маленький и быстрый русский
        # "model.hf_text.checkpoint_name": "DeepPavlov/rubert-base-cased",
        # "model.timm_image.checkpoint_name": "swinv2_large_window12_192_22k",
        # "env.num_gpus": 1,
    },
    
    time_limit=3600*6,
)

In [ ]:
predictor = MultiModalPredictor(label='sentiment')
predictor.fit(train_df, hyperparameters={"model.hf_text.checkpoint_name": "xlm-roberta-large"})

In [ ]:
predictor = MultiModalPredictor(label='breed')
predictor.fit(train_df, presets='best_quality')   # сам поймёт, что только image_path

In [ ]:
presets='best_quality'      # 4–12 часов, часто топ-1 на табличных + текст + фото
presets='high_quality_fast_finetune'   # быстрое дообучение, отличное качество
presets='multimodal_medium' # если мало времени/памяти

In [ ]:
# Посмотреть, какие модели вошли в ансамбль
print(predictor.leaderboard())

# Сохранить модель
predictor.save('my_best_multimodal_model')

# Загрузить позже
predictor = MultiModalPredictor.load('my_best_multimodal_model')